# Monitoring & Observability of AI Systems


This notebook demonstrates how to instrument, trace, evaluate, and monitor an agentic RAG system using **Ollama** for local inference and **LangSmith** for observability.

### Demo Outline
1. **Setup** — Ollama connection, LangSmith configuration, a simple RAG agent
2. **Manual Tracing** — Build a tracing framework from scratch to understand the concepts
3. **Core Metrics Collection** — Latency, cost, throughput, reliability
4. **Agent Decision Path Tracing** — Full trace of a multi-step agent run
5. **RAG Quality Monitoring** — Retrieval relevance and groundedness checks
6. **LangSmith Integration** — Real tracing with the LangSmith SDK
7. **Evaluation & Testing** — Running evals on production traces
8. **Alerting & Dashboards** — Setting up monitoring rules

---

## Part 0: Install Dependencies

Requires:
- Ollama running locally with a model pulled (e.g., `ollama pull llama3.1:latest`)
- A LangSmith API key (free tier at [smith.langchain.com](https://smith.langchain.com))

> **Note:** Parts 1–5 work without LangSmith. Parts 6–8 require a LangSmith account. You can demo the first half and then switch to LangSmith for the second half.

In [2]:
#!pip install -q requests rich langsmith langchain langchain-community

In [24]:
import os

# ── LangSmith Configuration ──
# Set these before running the LangSmith cells (Parts 6-8)
# Get your key at https://smith.langchain.com

import os

# Load environment variables from a keys.env file in the local folder if using python-dotenv
from dotenv import load_dotenv
load_dotenv('keys.env')

print("Environment configured.")
print(f"  LangSmith project: {os.getenv('LANGCHAIN_PROJECT')}")
print(f"  Tracing enabled:   {os.getenv('LANGCHAIN_TRACING_V2')}")


Environment configured.
  LangSmith project: segment7-monitoring-demo
  Tracing enabled:   true
  API Key:           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


---
## Part 1: Setup — Our Observable Agent

We'll build a RAG-style agent with tools, then progressively add observability layers.

In [5]:
import requests
import json
import re
import time
import uuid
import hashlib
from datetime import datetime
from collections import defaultdict
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.tree import Tree
from rich import print as rprint

console = Console()

# ── Ollama Configuration ──
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:latest"  # Change to your pulled model

def chat_raw(messages, temperature=0.1):
    """Send messages to Ollama, return full response with metadata."""
    start = time.time()
    resp = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature}
    })
    resp.raise_for_status()
    data = resp.json()
    elapsed = time.time() - start
    
    return {
        "content": data["message"]["content"],
        "model": data.get("model", MODEL),
        "total_duration_ns": data.get("total_duration", 0),
        "prompt_eval_count": data.get("prompt_eval_count", 0),
        "eval_count": data.get("eval_count", 0),
        "wall_time_s": elapsed,
    }

# Quick test
test = chat_raw([{"role": "user", "content": "Say 'connected' and nothing else."}])
console.print(Panel(
    f"{test['content']}\n\n[dim]Wall time: {test['wall_time_s']:.2f}s | "
    f"Prompt tokens: {test['prompt_eval_count']} | "
    f"Output tokens: {test['eval_count']}[/dim]",
    title="Ollama Connected", border_style="green"
))

╭─────────────────────────────────────────────── Ollama Connected ────────────────────────────────────────────────╮
│ Connected                                                                                                       │
│                                                                                                                 │
│ Wall time: 4.60s | Prompt tokens: 18 | Output tokens: 2                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
# ── Simulated RAG Knowledge Base ──

KNOWLEDGE_BASE = [
    {"id": "doc-001", "topic": "remote_work", "content": "Acme Corp allows employees to work remotely up to 3 days per week. Remote work requests must be approved by direct managers. Equipment stipend of $500 is provided annually."},
    {"id": "doc-002", "topic": "pto", "content": "All full-time employees receive 20 days of PTO per year. PTO accrues monthly at 1.67 days. Unused PTO rolls over up to 5 days. PTO requests require 2 weeks advance notice for periods over 3 days."},
    {"id": "doc-003", "topic": "benefits", "content": "Acme Corp offers health insurance (PPO and HMO), dental, vision, 401k with 4% match, life insurance at 2x salary, and an employee assistance program. Open enrollment is in November."},
    {"id": "doc-004", "topic": "performance", "content": "Performance reviews are conducted bi-annually in June and December. Reviews use a 1-5 scale. Employees rated 4+ are eligible for promotion. Ratings of 2 or below trigger a performance improvement plan."},
    {"id": "doc-005", "topic": "onboarding", "content": "New hires complete a 2-week onboarding program. Week 1 covers company culture, tools, and compliance. Week 2 is team-specific training. Buddy system pairs new hires with experienced employees."},
    {"id": "doc-006", "topic": "expenses", "content": "Business expenses over $50 require receipt submission. Travel requires pre-approval. Meals during travel are reimbursed up to $75/day. Expense reports must be submitted within 30 days."},
]

def retrieve_documents(query, top_k=2):
    """Simulated retrieval with keyword matching (stands in for vector search)."""
    scores = []
    query_lower = query.lower()
    for doc in KNOWLEDGE_BASE:
        # Simple keyword scoring (simulates cosine similarity)
        words = set(doc["content"].lower().split())
        query_words = set(query_lower.split())
        overlap = len(words & query_words)
        topic_bonus = 3 if doc["topic"] in query_lower else 0
        score = overlap + topic_bonus
        scores.append((doc, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    results = [{"doc": s[0], "score": s[1] / max(1, max(sc[1] for sc in scores))} for s in scores[:top_k]]
    return results

# ── Simulated Tools ──

EMPLOYEE_DB = {
    "E001": {"name": "Alice Martin", "department": "Engineering", "manager": "Dave Wilson", "start_date": "2021-03-15"},
    "E002": {"name": "Bob Chen", "department": "Data Science", "manager": "Eve Johnson", "start_date": "2022-08-01"},
    "E003": {"name": "Carol Davis", "department": "Sales", "manager": "Frank Lee", "start_date": "2020-01-10"},
}

def tool_lookup_employee(employee_id):
    if employee_id.strip('"\' ') in EMPLOYEE_DB:
        return json.dumps(EMPLOYEE_DB[employee_id.strip('"\' ')])
    return f"Employee {employee_id} not found."

def tool_calculate_pto(employee_id):
    """Simulates PTO balance calculation."""
    return json.dumps({"employee_id": employee_id.strip('"\' '), "pto_balance": 14.5, "pto_used": 5.5, "pto_total": 20})

TOOLS = {
    "lookup_employee": tool_lookup_employee,
    "calculate_pto": tool_calculate_pto,
}

console.print(Panel(
    f"Knowledge base: {len(KNOWLEDGE_BASE)} documents\nTools: {', '.join(TOOLS.keys())}\nEmployees: {', '.join(EMPLOYEE_DB.keys())}",
    title="RAG Agent Environment", border_style="blue"
))

╭───────────────────────────────────────────── RAG Agent Environment ─────────────────────────────────────────────╮
│ Knowledge base: 6 documents                                                                                     │
│ Tools: lookup_employee, calculate_pto                                                                           │
│ Employees: E001, E002, E003                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Part 2: Manual Tracing — Understanding the Concepts

Before using LangSmith, let's build a **tracing framework from scratch** so you understand what's happening under the hood. This is what tools like LangSmith automate for you.

In [7]:
# ── Span & Trace Framework ──

class Span:
    """A single operation within a trace (LLM call, retrieval, tool call, etc.)."""
    
    def __init__(self, name, span_type, parent=None):
        self.id = str(uuid.uuid4())[:8]
        self.name = name
        self.span_type = span_type  # "llm", "retrieval", "tool", "chain", "agent"
        self.parent = parent
        self.start_time = time.time()
        self.end_time = None
        self.input_data = None
        self.output_data = None
        self.metadata = {}
        self.children = []
        self.status = "running"
        
        if parent:
            parent.children.append(self)
    
    def end(self, status="success"):
        self.end_time = time.time()
        self.status = status
    
    @property
    def duration_ms(self):
        if self.end_time:
            return (self.end_time - self.start_time) * 1000
        return (time.time() - self.start_time) * 1000


class Trace:
    """A complete trace of an agent run, containing nested spans."""
    
    def __init__(self, name="agent_run"):
        self.id = str(uuid.uuid4())[:12]
        self.root = Span(name, "agent")
        self.all_spans = [self.root]
        self.metrics = {
            "total_tokens_in": 0,
            "total_tokens_out": 0,
            "llm_calls": 0,
            "tool_calls": 0,
            "retrieval_calls": 0,
        }
    
    def span(self, name, span_type, parent=None):
        """Create a new span within this trace."""
        s = Span(name, span_type, parent=parent or self.root)
        self.all_spans.append(s)
        return s
    
    def end(self):
        self.root.end()
    
    def display_tree(self):
        """Display the trace as a visual tree."""
        def build_tree(span, tree_node):
            for child in span.children:
                dur = f"{child.duration_ms:.0f}ms"
                status_icon = {"success": "✅", "error": "❌", "running": "⏳"}.get(child.status, "")
                type_color = {
                    "llm": "magenta", "retrieval": "cyan", "tool": "yellow",
                    "chain": "blue", "agent": "green"
                }.get(child.span_type, "white")
                
                label = f"[{type_color}]{child.name}[/{type_color}] [{type_color}]{dur}[/{type_color}] {status_icon}"
                
                # Add token info for LLM spans
                if child.span_type == "llm" and "tokens_in" in child.metadata:
                    label += f" [dim](in:{child.metadata['tokens_in']} out:{child.metadata['tokens_out']})[/dim]"
                
                # Add score for retrieval spans
                if child.span_type == "retrieval" and "top_score" in child.metadata:
                    label += f" [dim](relevance:{child.metadata['top_score']:.2f})[/dim]"
                
                child_node = tree_node.add(label)
                build_tree(child, child_node)
        
        tree = Tree(f"[bold green]Trace {self.id}[/bold green] — {self.root.name} [{self.root.duration_ms:.0f}ms total]")
        build_tree(self.root, tree)
        console.print(tree)
    
    def display_metrics(self):
        """Display aggregated trace metrics."""
        table = Table(title=f"Trace Metrics — {self.id}")
        table.add_column("Metric", style="cyan")
        table.add_column("Value", justify="right")
        
        table.add_row("Total Duration", f"{self.root.duration_ms:.0f} ms")
        table.add_row("LLM Calls", str(self.metrics["llm_calls"]))
        table.add_row("Tool Calls", str(self.metrics["tool_calls"]))
        table.add_row("Retrieval Calls", str(self.metrics["retrieval_calls"]))
        table.add_row("Total Input Tokens", str(self.metrics["total_tokens_in"]))
        table.add_row("Total Output Tokens", str(self.metrics["total_tokens_out"]))
        
        total_tokens = self.metrics["total_tokens_in"] + self.metrics["total_tokens_out"]
        # Approximate cost (using GPT-4o-mini pricing as reference)
        approx_cost = (self.metrics["total_tokens_in"] * 0.15 + self.metrics["total_tokens_out"] * 0.6) / 1_000_000
        table.add_row("Est. Cost (cloud equiv.)", f"${approx_cost:.6f}")
        
        console.print(table)

console.print(Panel("Tracing framework ready", title="Tracer", border_style="green"))

╭──────────────────────────────────────────────────── Tracer ─────────────────────────────────────────────────────╮
│ Tracing framework ready                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Note:

> This is fundamentally what LangSmith, LangFuse, and other tools do under the hood. A **trace** is a complete agent run. Inside, **spans** represent each operation — LLM calls, retrievals, tool calls. Spans nest inside each other to form a tree. Every span records its start time, end time, inputs, outputs, and metadata. Let's see it in action.

---
## Part 3: Traced Agent Run — Full Visibility

Now let's run our agent with full tracing enabled.

In [8]:
# ── Fully Traced RAG Agent ──

SYSTEM_PROMPT = """You are an HR assistant for Acme Corp. Answer questions using the provided context.
You have access to these tools:
- lookup_employee(employee_id): Look up employee details
- calculate_pto(employee_id): Calculate PTO balance

When you need a tool, respond with: TOOL_CALL: tool_name(argument)
When you have enough information, provide a helpful answer based on the retrieved context."""

def run_traced_agent(user_query):
    """Run the RAG agent with full tracing."""
    trace = Trace("rag_agent_run")
    trace.root.input_data = user_query
    
    # ── Step 1: Retrieval ──
    retrieval_span = trace.span("RAG Retrieval", "retrieval")
    retrieval_span.input_data = user_query
    
    # Vector search sub-span
    search_span = trace.span("Vector Search", "retrieval", parent=retrieval_span)
    search_span.input_data = user_query
    results = retrieve_documents(user_query, top_k=2)
    search_span.output_data = [r["doc"]["id"] for r in results]
    search_span.metadata["num_results"] = len(results)
    search_span.metadata["top_score"] = results[0]["score"] if results else 0
    search_span.end()
    
    # Build context
    context = "\n\n".join([f"[{r['doc']['id']}] {r['doc']['content']}" for r in results])
    retrieval_span.output_data = context[:200] + "..."
    retrieval_span.metadata["top_score"] = results[0]["score"] if results else 0
    retrieval_span.metadata["num_docs"] = len(results)
    trace.metrics["retrieval_calls"] += 1
    retrieval_span.end()
    
    # ── Step 2: LLM Call — Reasoning ──
    llm_span = trace.span("LLM — Reasoning", "llm")
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_query}"}
    ]
    llm_span.input_data = f"system_prompt + context ({len(context)} chars) + query"
    
    llm_result = chat_raw(messages)
    response = llm_result["content"]
    
    llm_span.output_data = response[:200] + "..." if len(response) > 200 else response
    llm_span.metadata["tokens_in"] = llm_result["prompt_eval_count"]
    llm_span.metadata["tokens_out"] = llm_result["eval_count"]
    llm_span.metadata["model"] = llm_result["model"]
    trace.metrics["total_tokens_in"] += llm_result["prompt_eval_count"]
    trace.metrics["total_tokens_out"] += llm_result["eval_count"]
    trace.metrics["llm_calls"] += 1
    llm_span.end()
    
    # ── Step 3: Tool Execution (if requested) ──
    if "TOOL_CALL:" in response:
        tool_match = re.search(r'TOOL_CALL:\s*(\w+)\((.*)\)', response)
        if tool_match:
            tool_name = tool_match.group(1)
            tool_args = tool_match.group(2).strip()
            
            tool_span = trace.span(f"Tool — {tool_name}", "tool")
            tool_span.input_data = f"{tool_name}({tool_args})"
            
            if tool_name in TOOLS:
                tool_result = TOOLS[tool_name](tool_args)
                tool_span.output_data = tool_result
                tool_span.end()
                trace.metrics["tool_calls"] += 1
                
                # ── Step 4: LLM Call — Final synthesis ──
                synth_span = trace.span("LLM — Synthesis", "llm")
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Tool result: {tool_result}"})
                synth_span.input_data = f"conversation + tool result"
                
                synth_result = chat_raw(messages)
                response = synth_result["content"]
                
                synth_span.output_data = response[:200] + "..." if len(response) > 200 else response
                synth_span.metadata["tokens_in"] = synth_result["prompt_eval_count"]
                synth_span.metadata["tokens_out"] = synth_result["eval_count"]
                trace.metrics["total_tokens_in"] += synth_result["prompt_eval_count"]
                trace.metrics["total_tokens_out"] += synth_result["eval_count"]
                trace.metrics["llm_calls"] += 1
                synth_span.end()
            else:
                tool_span.output_data = f"Unknown tool: {tool_name}"
                tool_span.end("error")
    
    trace.root.output_data = response
    trace.end()
    
    return response, trace

In [9]:
# ── Run a simple RAG query ──
console.print("\n[bold cyan]━━━ Traced RAG Query ━━━[/bold cyan]\n")

response, trace = run_traced_agent("What is the PTO policy at Acme Corp? How many days do I get?")

console.print(Panel(response, title="💬 Agent Response", border_style="green"))
console.print()
trace.display_tree()
console.print()
trace.display_metrics()

━━━ Traced RAG Query ━━━

╭─────────────────────────────────────────────── 💬 Agent Response ───────────────────────────────────────────────╮
│ Since the employee ID is not found, I'll provide the general PTO policy details.                                │
│                                                                                                                 │
│ According to our company policies, all full-time employees receive 20 days of PTO per year. PTO accrues monthly │
│ at 1.67 days. Unused PTO rolls over up to 5 days.                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Trace 8102e9ec-7e7 — rag_agent_run [6879ms total]
├── RAG Retrieval 0ms ✅ (relevance:1.00)
│   └── Vector Search 0ms ✅ (relevance:1.00)
├── LLM — Reasoning 3090ms ✅ (in:215 out:30)
├── Tool — lookup_employee 0ms ✅
└── LLM — Synthesis 3788ms ✅ (in:263 out:62)

      Trace Metrics — 8102e9ec-7e7      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric                   ┃     Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ Total Duration           │   6879 ms │
│ LLM Calls                │         2 │
│ Tool Calls               │         1 │
│ Retrieval Calls          │         1 │
│ Total Input Tokens       │       478 │
│ Total Output Tokens      │        92 │
│ Est. Cost (cloud equiv.) │ $0.000127 │
└──────────────────────────┴───────────┘

In [10]:
# ── Run a query that triggers a tool call ──
console.print("\n[bold cyan]━━━ Traced Agent Query (with Tool Call) ━━━[/bold cyan]\n")

response2, trace2 = run_traced_agent("What is employee E001's department and who is their manager?")

console.print(Panel(response2, title="💬 Agent Response", border_style="green"))
console.print()
trace2.display_tree()
console.print()
trace2.display_metrics()

━━━ Traced Agent Query (with Tool Call) ━━━

╭─────────────────────────────────────────────── 💬 Agent Response ───────────────────────────────────────────────╮
│ It seems that employee E001 is not in the system. I'll need more information to answer your question. Can you   │
│ please provide more context or details about employee E001?                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Trace d2498c87-567 — rag_agent_run [3772ms total]
├── RAG Retrieval 0ms ✅ (relevance:1.00)
│   └── Vector Search 0ms ✅ (relevance:1.00)
├── LLM — Reasoning 1451ms ✅ (in:207 out:13)
├── Tool — lookup_employee 0ms ✅
└── LLM — Synthesis 2321ms ✅ (in:242 out:36)

      Trace Metrics — d2498c87-567      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric                   ┃     Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ Total Duration           │   3772 ms │
│ LLM Calls                │         2 │
│ Tool Calls               │         1 │
│ Retrieval Calls          │         1 │
│ Total Input Tokens       │       449 │
│ Total Output Tokens      │        49 │
│ Est. Cost (cloud equiv.) │ $0.000097 │
└──────────────────────────┴───────────┘

### Note:

> Notce this tree structure. This is exactly what you see in LangSmith, LangFuse, or any tracing tool. The tree shows you the full decision path — retrieval, then reasoning, then optionally a tool call and synthesis. Each node shows its duration, token counts, and status. You can immediately see where the time went. If the retrieval step was slow, you'd see it here. If the LLM call was expensive, the token count tells you why.

---
## Part 4: Core Metrics Dashboard

Let's run multiple queries and collect operational metrics — latency, cost, throughput, reliability.

In [11]:
# ── Metrics Collector ──

class MetricsCollector:
    """Collects and aggregates operational metrics across runs."""
    
    def __init__(self):
        self.traces = []
        self.latencies = []       # total ms per run
        self.token_costs = []     # estimated cost per run
        self.success_count = 0
        self.error_count = 0
        self.total_tokens_in = 0
        self.total_tokens_out = 0
    
    def record(self, trace):
        self.traces.append(trace)
        self.latencies.append(trace.root.duration_ms)
        
        cost = (trace.metrics["total_tokens_in"] * 0.15 + trace.metrics["total_tokens_out"] * 0.6) / 1_000_000
        self.token_costs.append(cost)
        
        self.total_tokens_in += trace.metrics["total_tokens_in"]
        self.total_tokens_out += trace.metrics["total_tokens_out"]
        
        has_error = any(s.status == "error" for s in trace.all_spans)
        if has_error:
            self.error_count += 1
        else:
            self.success_count += 1
    
    def display_dashboard(self):
        """Display a production-style metrics dashboard."""
        import statistics
        
        if not self.latencies:
            console.print("[dim]No data yet.[/dim]")
            return
        
        n = len(self.latencies)
        sorted_lat = sorted(self.latencies)
        p50 = sorted_lat[int(n * 0.5)] if n > 1 else sorted_lat[0]
        p95 = sorted_lat[int(n * 0.95)] if n > 1 else sorted_lat[0]
        p99 = sorted_lat[min(int(n * 0.99), n-1)]
        
        total = self.success_count + self.error_count
        success_rate = (self.success_count / total * 100) if total > 0 else 0
        
        console.print("\n[bold]━━━ OPERATIONAL DASHBOARD ━━━[/bold]\n")
        
        # Latency panel
        lat_table = Table(title="Latency", show_header=True)
        lat_table.add_column("P50", justify="center", style="green")
        lat_table.add_column("P95", justify="center", style="yellow")
        lat_table.add_column("P99", justify="center", style="red")
        lat_table.add_column("Avg", justify="center")
        lat_table.add_row(
            f"{p50:.0f}ms", f"{p95:.0f}ms", f"{p99:.0f}ms",
            f"{statistics.mean(self.latencies):.0f}ms"
        )
        console.print(lat_table)
        
        # Cost panel
        cost_table = Table(title="Cost (Cloud Equivalent)", show_header=True)
        cost_table.add_column("Total Runs", justify="center")
        cost_table.add_column("Avg Cost/Run", justify="center")
        cost_table.add_column("Total Cost", justify="center")
        cost_table.add_column("Total Tokens", justify="center")
        cost_table.add_row(
            str(n),
            f"${statistics.mean(self.token_costs):.6f}",
            f"${sum(self.token_costs):.6f}",
            f"{self.total_tokens_in + self.total_tokens_out:,}"
        )
        console.print(cost_table)
        
        # Reliability panel
        rel_table = Table(title="Reliability", show_header=True)
        rel_table.add_column("Success Rate", justify="center")
        rel_table.add_column("Successes", justify="center", style="green")
        rel_table.add_column("Errors", justify="center", style="red")
        rel_table.add_row(
            f"{success_rate:.1f}%",
            str(self.success_count),
            str(self.error_count)
        )
        console.print(rel_table)
        
        # Per-trace breakdown
        breakdown = Table(title="Per-Run Breakdown")
        breakdown.add_column("#", style="dim")
        breakdown.add_column("Query", max_width=40)
        breakdown.add_column("Latency", justify="right")
        breakdown.add_column("LLM Calls", justify="center")
        breakdown.add_column("Tool Calls", justify="center")
        breakdown.add_column("Tokens", justify="right")
        breakdown.add_column("Cost", justify="right")
        
        for i, t in enumerate(self.traces):
            query = str(t.root.input_data or "")[:40]
            total_tok = t.metrics["total_tokens_in"] + t.metrics["total_tokens_out"]
            breakdown.add_row(
                str(i+1), query,
                f"{t.root.duration_ms:.0f}ms",
                str(t.metrics["llm_calls"]),
                str(t.metrics["tool_calls"]),
                f"{total_tok:,}",
                f"${self.token_costs[i]:.6f}"
            )
        
        console.print(breakdown)

In [12]:
# ── Run multiple queries and collect metrics ──

queries = [
    "What is the PTO policy?",
    "Tell me about health benefits and 401k matching.",
    "What department is employee E002 in?",
    "How does the performance review process work?",
    "What is the expense reimbursement limit for meals during travel?",
    "What is employee E001's start date and who is their manager?",
    "How does the onboarding buddy system work?",
    "How many days per week can I work remotely?",
]

metrics = MetricsCollector()

for i, q in enumerate(queries):
    console.print(f"[dim]Running query {i+1}/{len(queries)}: {q[:50]}...[/dim]")
    response, trace = run_traced_agent(q)
    metrics.record(trace)

metrics.display_dashboard()

Running query 1/8: What is the PTO policy?...

Running query 2/8: Tell me about health benefits and 401k matching....

Running query 3/8: What department is employee E002 in?...

Running query 4/8: How does the performance review process work?...

Running query 5/8: What is the expense reimbursement limit for meals ...

...

━━━ OPERATIONAL DASHBOARD ━━━

               Latency               
┏━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃  P50   ┃  P95   ┃  P99   ┃  Avg   ┃
┡━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ 5633ms │ 9175ms │ 9175ms │ 5254ms │
└────────┴────────┴────────┴────────┘

                 Cost (Cloud Equivalent)                 
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Total Runs ┃ Avg Cost/Run ┃ Total Cost ┃ Total Tokens ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│     8      │  $0.000085   │ $0.000684  │    2,702     │
└────────────┴──────────────┴────────────┴──────────────┘

             Reliability             
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Success Rate ┃ Successes ┃ Errors ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│    100.0%    │     8     │   0    │
└──────────────┴───────────┴────────┘

                                           Per-Run Breakdown                                            
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┓
┃ # ┃ Query                                    ┃ Latency ┃ LLM Calls ┃ Tool Calls ┃ Tokens ┃      Cost ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━┩
│ 1 │ What is the PTO policy?                  │  5633ms │     1     │     0      │    287 │ $0.000080 │
│ 2 │ Tell me about health benefits and 401k m │  7700ms │     1     │     0      │    334 │ $0.000106 │
│ 3 │ What department is employee E002 in?     │  3446ms │     2     │     1      │    483 │ $0.000096 │
│ 4 │ How does the performance review process  │  8050ms │     1     │     0      │    319 │ $0.000103 │
│ 5 │ What is the expense reimbursement limit  │  1923ms │     1     │     0      │    212 │ $0.000041 │
│ 6 │ What is employee E001's start date and w │  3882ms │     2     │     1      │    500 │ $0.000097 │
│ 7 │ How does the onboarding buddy system wor │  9175ms │     1     │     0      │    335 │ $0.000114 │
│ 8 │ How many days per week can I work remote │  2221ms │     1     │     0      │    232 │ $0.000046 │
└───┴──────────────────────────────────────────┴─────────┴───────────┴────────────┴────────┴───────────┘

### Note:

> Look at the dashboard. This is what a production monitoring dashboard looks like. P50 is your typical user experience. P95 is your slow users. P99 is your worst case. Notice how queries with tool calls are slower — that extra LLM roundtrip adds latency. The cost column shows the cloud-equivalent price. In production, these numbers compound fast — multiply by thousands of daily requests."

---
## Part 5: RAG Quality Monitoring

Operational metrics tell you the system is *running*. Quality metrics tell you it's *working correctly*.

In [13]:
# ── RAG Quality Evaluator ──

def evaluate_retrieval_relevance(query, retrieved_docs):
    """Use the LLM to judge whether retrieved documents are relevant."""
    doc_text = "\n".join([f"Doc {r['doc']['id']}: {r['doc']['content'][:100]}..." for r in retrieved_docs])
    
    messages = [{
        "role": "system",
        "content": """You are a relevance judge. Given a query and retrieved documents, rate how relevant each document is.
Respond with ONLY a JSON object: {"relevance_score": 0.0-1.0, "reasoning": "brief explanation"}"""
    }, {
        "role": "user",
        "content": f"Query: {query}\n\nRetrieved documents:\n{doc_text}"
    }]
    
    result = chat_raw(messages)
    try:
        match = re.search(r'\{[^}]+\}', result["content"])
        if match:
            return json.loads(match.group())
    except:
        pass
    return {"relevance_score": 0.5, "reasoning": "Could not parse evaluation"}


def evaluate_groundedness(query, context, response):
    """Check if the response is grounded in the provided context."""
    messages = [{
        "role": "system",
        "content": """You are a groundedness checker. Given a context and a response, determine if every claim in the response is supported by the context.
Respond with ONLY a JSON object: {"groundedness_score": 0.0-1.0, "unsupported_claims": ["list any claims not in context"], "reasoning": "brief explanation"}"""
    }, {
        "role": "user",
        "content": f"Context:\n{context}\n\nResponse:\n{response}"
    }]
    
    result = chat_raw(messages)
    try:
        match = re.search(r'\{[^}]+\}', result["content"], re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    return {"groundedness_score": 0.5, "unsupported_claims": [], "reasoning": "Could not parse"}

In [14]:
# ── Run quality evaluations ──

eval_queries = [
    "What is the PTO policy?",
    "How does the 401k match work?",
    "What happens if I get a low performance rating?",
    "What is the company's stance on cryptocurrency?",  # Not in knowledge base
]

quality_table = Table(title="RAG Quality Evaluation")
quality_table.add_column("Query", max_width=40)
quality_table.add_column("Retrieval\nRelevance", justify="center")
quality_table.add_column("Groundedness", justify="center")
quality_table.add_column("Notes", max_width=40)

for q in eval_queries:
    console.print(f"[dim]Evaluating: {q[:50]}...[/dim]")
    
    # Run the agent
    docs = retrieve_documents(q, top_k=2)
    context = "\n".join([r["doc"]["content"] for r in docs])
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {q}"}
    ]
    resp = chat_raw(messages)
    
    # Evaluate
    relevance = evaluate_retrieval_relevance(q, docs)
    groundedness = evaluate_groundedness(q, context, resp["content"])
    
    rel_score = relevance.get("relevance_score", 0)
    gnd_score = groundedness.get("groundedness_score", 0)
    
    rel_color = "green" if rel_score > 0.7 else "yellow" if rel_score > 0.4 else "red"
    gnd_color = "green" if gnd_score > 0.7 else "yellow" if gnd_score > 0.4 else "red"
    
    unsupported = groundedness.get("unsupported_claims", [])
    notes = "; ".join(unsupported) if unsupported else relevance.get("reasoning", "")
    
    quality_table.add_row(
        q[:40],
        f"[{rel_color}]{rel_score:.2f}[/{rel_color}]",
        f"[{gnd_color}]{gnd_score:.2f}[/{gnd_color}]",
        notes[:40]
    )

console.print(quality_table)

Evaluating: What is the PTO policy?...

Evaluating: How does the 401k match work?...

Evaluating: What happens if I get a low performance rating?...

Evaluating: What is the company's stance on cryptocurrency?...

                                              RAG Quality Evaluation                                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                          ┃ Retrieval ┃              ┃                                          ┃
┃ Query                                    ┃ Relevance ┃ Groundedness ┃ Notes                                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ What is the PTO policy?                  │   0.80    │     1.00     │ Doc doc-002 directly answers the query b │
│ How does the 401k match work?            │   0.80    │     1.00     │ The document mentions the 401k plan with │
│ What happens if I get a low performance  │   0.80    │     1.00     │ Doc doc-004 directly addresses performan │
│ What is the company's stance on cryptocu │   0.00    │     1.00     │ Neither document mentions cryptocurrency │
└──────────────────────────────────────────┴───────────┴──────────────┴──────────────────────────────────────────┘

### Note:

> The cryptocurrency question has no matching content in our knowledge base. Watch the retrieval relevance score — it should be low. If the groundedness score is also low, that means the model is hallucinating rather than saying 'I don't know.' This is exactly the kind of silent failure that operational metrics would never catch. The system returned a 200 OK, latency was normal, but the answer was wrong.

---
## Part 6: LangSmith Integration

Now let's switch from our manual tracing to **LangSmith** — seeing the same concepts in a production-grade tool.

>  **Requires a LangSmith API key.** Set it in Part 0 above.

In [16]:
# ── LangSmith Tracing with the SDK ──

from langsmith import traceable, Client
from langsmith.run_helpers import get_current_run_tree

ls_client = Client()

# Verify connection
try:
    projects = list(ls_client.list_projects(limit=1))
    console.print(Panel("LangSmith connected!", title="LangSmith", border_style="green"))
except Exception as e:
    console.print(Panel(f"LangSmith not configured: {e}\n\nSet LANGCHAIN_API_KEY in Part 0.",
                        title="LangSmith", border_style="yellow"))

╭─────────────────────────────────────────────────── LangSmith ───────────────────────────────────────────────────╮
│ LangSmith connected!                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
# ── Traced functions using @traceable ──
# Every function decorated with @traceable automatically appears as a span in LangSmith

@traceable(name="retrieve_documents", run_type="retriever")
def ls_retrieve(query, top_k=2):
    """Retrieve documents — automatically traced by LangSmith."""
    results = retrieve_documents(query, top_k)
    return [{"id": r["doc"]["id"], "content": r["doc"]["content"], "score": r["score"]} for r in results]

@traceable(name="ollama_chat", run_type="llm")
def ls_chat(messages, temperature=0.1):
    """LLM call — automatically traced by LangSmith."""
    result = chat_raw(messages, temperature)
    return result

@traceable(name="tool_call", run_type="tool")
def ls_tool_call(tool_name, tool_args):
    """Tool execution — automatically traced by LangSmith."""
    if tool_name in TOOLS:
        return TOOLS[tool_name](tool_args)
    return f"Unknown tool: {tool_name}"

@traceable(name="rag_agent", run_type="chain")
def ls_agent(query):
    """Full RAG agent — automatically traced by LangSmith."""
    
    # Step 1: Retrieve
    docs = ls_retrieve(query)
    context = "\n\n".join([f"[{d['id']}] {d['content']}" for d in docs])
    
    # Step 2: Reason
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
    result = ls_chat(messages)
    response = result["content"]
    
    # Step 3: Tool call if needed
    if "TOOL_CALL:" in response:
        tool_match = re.search(r'TOOL_CALL:\s*(\w+)\((.*)\)', response)
        if tool_match:
            tool_result = ls_tool_call(tool_match.group(1), tool_match.group(2).strip())
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": f"Tool result: {tool_result}"})
            result = ls_chat(messages)
            response = result["content"]
    
    return response

console.print(Panel("LangSmith-traced agent ready. Every call will appear in your LangSmith dashboard.",
                     title="LangSmith Agent", border_style="cyan"))

╭──────────────────────────────────────────────── LangSmith Agent ────────────────────────────────────────────────╮
│ LangSmith-traced agent ready. Every call will appear in your LangSmith dashboard.                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [18]:
# ── Run queries through the LangSmith-traced agent ──

ls_queries = [
    "What is the PTO policy at Acme Corp?",
    "What department is employee E001 in?",
    "How does the performance review process work? What happens with a rating of 2?",
    "What is the expense limit for meals during business travel?",
]

for q in ls_queries:
    console.print(f"\n[bold cyan]Query:[/bold cyan] {q}")
    response = ls_agent(q)
    console.print(f"[green]Response:[/green] {response[:200]}{'...' if len(response) > 200 else ''}\n")

console.print(Panel(
    "All traces sent to LangSmith!\n\n"
    "Open your LangSmith dashboard to see:\n"
    "  → Run tree with nested spans\n"
    "  → Input/output at every step\n"
    "  → Latency waterfall\n"
    "  → Token counts per LLM call",
    title="Check LangSmith Dashboard", border_style="cyan"
))

Query: What is the PTO policy at Acme Corp?

Response: Based on the provided context, the PTO policy at Acme Corp is as follows:

* Full-time employees receive 20 days of PTO per year.
* PTO accrues monthly at a rate of 1.67 days.
* Unused PTO rolls over ...

Query: What department is employee E001 in?

Response: It seems that employee E001 is not in our system. I'll need more information to answer your question. Can
you please provide more context or details about employee E001?

Query: How does the performance review process work? What happens with a rating of 2?

Response: Based on the provided context, here's how the performance review process works:

Performance reviews are conducted bi-annually in June and December. During these reviews, employees are rated on a 
1-5 ...

Query: What is the expense limit for meals during business travel?

Response: Based on the provided context, the expense limit for meals during business travel is $75 per day.

╭───────────────────────────────────────── 🔗 Check LangSmith Dashboard ──────────────────────────────────────────╮
│ All traces sent to LangSmith!                                                                                   │
│                                                                                                                 │
│ Open your LangSmith dashboard to see:                                                                           │
│   → Run tree with nested spans                                                                                  │
│   → Input/output at every step                                                                                  │
│   → Latency waterfall                                                                                           │
│   → Token counts per LLM call                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Note:

> **Now you can switch your screen to the LangSmith web UI.Things to see:
> 1. The **project view** showing all runs
> 2. Click into a run to see the **trace tree** (same concept as our manual tree, but richer)
> 3. Show **input/output** at each node
> 4. Show the **latency waterfall**
> 5. Show **metadata and token counts**
>
> Compare this to the manual trace tree we built earlier. It's the same concept — spans nested in a trace — but LangSmith gives you a searchable, filterable, production-grade interface. And all we did was add `@traceable` decorators to our functions.

---
## Part 7: Evaluation with LangSmith

Let's create a test dataset and run evaluations — the workflow from Slide 17.

In [19]:
# ── Step 1: Create a test dataset ──

dataset_name = "hr-agent-eval-demo"

# Delete if exists (for re-runs)
try:
    existing = ls_client.read_dataset(dataset_name=dataset_name)
    ls_client.delete_dataset(dataset_id=existing.id)
    console.print(f"[dim]Deleted existing dataset '{dataset_name}'[/dim]")
except:
    pass

dataset = ls_client.create_dataset(dataset_name, description="HR Agent evaluation test cases")

# Add test cases
test_cases = [
    {
        "input": {"query": "How many PTO days do full-time employees get?"},
        "expected": {"answer": "20 days per year", "should_contain": ["20"]}
    },
    {
        "input": {"query": "What is the 401k employer match?"},
        "expected": {"answer": "4% match", "should_contain": ["4%"]}
    },
    {
        "input": {"query": "How many days per week can I work remotely?"},
        "expected": {"answer": "Up to 3 days per week", "should_contain": ["3"]}
    },
    {
        "input": {"query": "What happens if I receive a performance rating of 2?"},
        "expected": {"answer": "Performance improvement plan", "should_contain": ["improvement", "plan"]}
    },
    {
        "input": {"query": "What is the meal reimbursement limit during travel?"},
        "expected": {"answer": "$75 per day", "should_contain": ["75"]}
    },
]

for tc in test_cases:
    ls_client.create_example(
        inputs=tc["input"],
        outputs=tc["expected"],
        dataset_id=dataset.id
    )

console.print(Panel(
    f"Created dataset '{dataset_name}' with {len(test_cases)} test cases",
    title="Evaluation Dataset", border_style="cyan"
))

╭────────────────────────────────────────────── Evaluation Dataset ───────────────────────────────────────────────╮
│ Created dataset 'hr-agent-eval-demo' with 5 test cases                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [21]:
# ── Step 2: Define evaluators ──

from langsmith.evaluation import evaluate, EvaluationResult

def contains_expected(run, example) -> EvaluationResult:
    """Check if the response contains expected keywords."""
    response = (run.outputs or {}).get("output", "").lower()
    expected_terms = (example.outputs or {}).get("should_contain", [])
    
    if not expected_terms:
        return EvaluationResult(key="contains_expected", score=1.0)
    
    matches = sum(1 for term in expected_terms if term.lower() in response)
    score = matches / len(expected_terms)
    
    return EvaluationResult(
        key="contains_expected",
        score=score,
        comment=f"Matched {matches}/{len(expected_terms)} expected terms"
    )

def response_length_check(run, example) -> EvaluationResult:
    """Check that the response is a reasonable length (not too short, not too long)."""
    response = (run.outputs or {}).get("output", "")
    length = len(response)
    
    if length < 20:
        score = 0.2  # Too short — probably an error
    elif length > 2000:
        score = 0.5  # Too verbose
    else:
        score = 1.0
    
    return EvaluationResult(
        key="response_length",
        score=score,
        comment=f"Length: {length} chars"
    )

console.print(Panel("Evaluators defined: contains_expected, response_length_check",
                     title="Evaluators", border_style="cyan"))

╭────────────────────────────────────────────────── Evaluators ───────────────────────────────────────────────────╮
│ Evaluators defined: contains_expected, response_length_check                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [22]:
# ── Step 3: Run the evaluation ──

def agent_target(inputs: dict) -> dict:
    """Wrapper for evaluation — takes inputs dict, returns outputs dict."""
    query = inputs["query"]
    response = ls_agent(query)
    return {"output": response}

console.print("\n[bold cyan]Running evaluation...[/bold cyan]")
console.print("[dim]This runs each test case through the agent and scores the results.[/dim]\n")

results = evaluate(
    agent_target,
    data=dataset_name,
    evaluators=[contains_expected, response_length_check],
    experiment_prefix="hr-agent-v1",
)

console.print(Panel(
    "Evaluation complete!\n\n"
    "Open LangSmith to see:\n"
    "  → Per-example scores\n"
    "  → Aggregate pass/fail rates\n"
    "  → Full traces for each test case\n"
    "  → Compare against future experiments",
    title="Evaluation Results in LangSmith", border_style="green"
))

Running evaluation...

This runs each test case through the agent and scores the results.

View the evaluation results for experiment: 'hr-agent-v1-005fa3b0' at:
https://smith.langchain.com/o/02c290f5-ee94-4eff-8d2a-27dbdde8ced1/datasets/3938ca46-d486-40b0-af3c-1e25d477213d/compare?selectedSessions=e8bfbe36-aa48-4772-80cb-6fadb7d52a9a




0it [00:00, ?it/s]

╭──────────────────────────────────────── Evaluation Results in LangSmith ────────────────────────────────────────╮
│ Evaluation complete!                                                                                            │
│                                                                                                                 │
│ Open LangSmith to see:                                                                                          │
│   → Per-example scores                                                                                          │
│   → Aggregate pass/fail rates                                                                                   │
│   → Full traces for each test case                                                                              │
│   → Compare against future experiments                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Note:

> After you run this cell, **switch to the LangSmith UI** and navigate to the Datasets & Testing section.
> 
> Look at:
> 1. The **dataset** with its test cases
> 2. The **experiment results** showing per-case scores
> 3. **Aggregate metrics** — what percentage of tests passed?
> 4. Click into a failing case to see the **full trace** and understand why
>
> This is how you catch regressions. Next time you change a prompt or swap a model, run this same evaluation. If scores drop, you know before deploying. Every production failure should become a test case in this dataset.

---
## Part 8: Alerting Concepts

Finally, let's demonstrate how you'd set up alerting rules to catch production issues.

In [23]:
# ── Alert Rule Engine ──

class AlertEngine:
    """Simulates production alerting based on metrics thresholds."""
    
    def __init__(self):
        self.rules = []
        self.alerts_fired = []
    
    def add_rule(self, name, level, check_fn, message_fn):
        self.rules.append({"name": name, "level": level, "check": check_fn, "message": message_fn})
    
    def evaluate(self, metrics_collector):
        """Run all rules against collected metrics."""
        self.alerts_fired = []
        for rule in self.rules:
            triggered, value = rule["check"](metrics_collector)
            if triggered:
                alert = {
                    "rule": rule["name"],
                    "level": rule["level"],
                    "message": rule["message"](value),
                    "timestamp": datetime.now().isoformat(),
                }
                self.alerts_fired.append(alert)
        return self.alerts_fired
    
    def display(self):
        if not self.alerts_fired:
            console.print(Panel("[green]All clear — no alerts triggered.[/green]",
                                title="Alert Status", border_style="green"))
            return
        
        table = Table(title="Alerts Fired")
        table.add_column("Level", justify="center")
        table.add_column("Rule", style="cyan")
        table.add_column("Message")
        
        for alert in self.alerts_fired:
            level_style = {"critical": "bold red", "warning": "yellow", "info": "blue"}.get(alert["level"], "white")
            table.add_row(
                f"[{level_style}]{alert['level'].upper()}[/{level_style}]",
                alert["rule"],
                alert["message"]
            )
        
        console.print(table)


# ── Define alert rules ──
engine = AlertEngine()

engine.add_rule(
    "high_p95_latency", "warning",
    lambda m: (sorted(m.latencies)[int(len(m.latencies)*0.95)] > 5000, sorted(m.latencies)[int(len(m.latencies)*0.95)]) if m.latencies else (False, 0),
    lambda v: f"P95 latency is {v:.0f}ms (threshold: 5000ms)"
)

engine.add_rule(
    "error_rate", "critical",
    lambda m: ((m.error_count / max(1, m.success_count + m.error_count)) > 0.1, m.error_count / max(1, m.success_count + m.error_count)),
    lambda v: f"Error rate is {v*100:.1f}% (threshold: 10%)"
)

engine.add_rule(
    "cost_per_run", "warning",
    lambda m: (any(c > 0.01 for c in m.token_costs), max(m.token_costs) if m.token_costs else 0),
    lambda v: f"Run cost ${v:.6f} exceeds threshold $0.01"
)

engine.add_rule(
    "high_token_usage", "info",
    lambda m: (m.total_tokens_in + m.total_tokens_out > 5000, m.total_tokens_in + m.total_tokens_out),
    lambda v: f"Session token usage {v:,} exceeds 5,000 token threshold"
)

# Run alerts against our collected metrics
alerts = engine.evaluate(metrics)
engine.display()

console.print("\n[dim]In production, these alerts would fire to Slack, PagerDuty, or your SIEM.[/dim]")

                               Alerts Fired                               
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  Level  ┃ Rule             ┃ Message                                   ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ WARNING │ high_p95_latency │ P95 latency is 9175ms (threshold: 5000ms) │
└─────────┴──────────────────┴───────────────────────────────────────────┘

In production, these alerts would fire to Slack, PagerDuty, or your SIEM.

### Note:

> These are the AI-specific alerts from Slide 13. Notice we're alerting on latency percentiles, cost per run, error rates, and token usage — not just HTTP status codes. In production, you'd add alerts for hallucination rate spikes, guardrail block rate changes, and quality score drift. LangSmith provides built-in monitoring rules for many of these.

---
## Summary: What We Built

| Part | What We Did | Slide Connection |
|------|------------|------------------|
| **Manual Tracing** | Built spans and traces from scratch | Slides 10-11: Agent Decision Paths |
| **Core Metrics** | Collected latency, cost, throughput, reliability | Slides 4-7: Core Metrics |
| **Quality Monitoring** | Evaluated retrieval relevance and groundedness | Slides 9, 12: Model Behavior & RAG |
| **LangSmith Tracing** | Used `@traceable` for automatic instrumentation | Slides 15-16: LangSmith Tracing |
| **LangSmith Evaluation** | Created datasets and ran scored experiments | Slide 17: Evaluation & Testing |
| **Alerting** | Defined rules for AI-specific metrics | Slide 13: Alerting Strategies |

### Key Principles Demonstrated

1. **Trace everything** — Every LLM call, retrieval, and tool call becomes a span in a trace
2. **Four pillars** — Latency, cost, throughput, reliability tracked for every run
3. **Quality is a metric** — Retrieval relevance and groundedness scored automatically
4. **From manual to production** — Same concepts, but LangSmith automates it with `@traceable`
5. **Evaluation as CI** — Test datasets catch regressions before they reach users
6. **AI-specific alerting** — Alert on semantics, not just status codes

### That's a Wrap!

You now have the tools and techniques to deploy, secure, **and observe** AI systems in production. Go build something great — and make sure you can see what it's doing.